# 82 — ColBERT conversational retrieval: zero-shot pool-rerank probe (P1 kill-gate)

Mirrors nb74's recall harness. **Phase 1 (plan §9):** use off-the-shelf
`colbert-ir/colbertv2.0` as a *pool reranker* over the config-200 union pool
(`cs`) on the **dev turn-1 subset** — no PLAID index, no fine-tune. 

**Kill-gate:** ColBERT must beat single-vector dense on **turn-1 wall recall@20**.
If it does not, late interaction does not transfer to this domain → STOP, bank
config 200. Cells 1/3/4/5 are copied verbatim from nb74; the new work is the
`#82-probe` and `#82-gate` cells.

In [ ]:
# 1) Setup. Disable JAX GPU preallocation BEFORE any import pulls JAX in
# (datasets/transformers import JAX transitively; it grabs ~75% VRAM on first use).
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'recall-union-lgbm'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

# Symlink the persistent caches from Drive. retrieval_v2 holds sasrec/, lgbm/,
# ctx_cache/; dense holds the Qwen query/catalog cache for dense_metadata_qwen3.
DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, drive_subdir in [
    ('retrieval_v2', 'recsys2026_retrieval_v2_cache'),
    ('dense', 'recsys2026_dense_cache'),
]:
    src = f'{DRIVE_BASE}/{drive_subdir}'
    dst = f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

# Deps: retrieval stack + ColBERT late-interaction (pylate). pylate is the only
# addition vs nb74 (lazy-imported by mcrs.retrieval_modules.colbert_late).
!pip install -q --upgrade 'transformers>=4.40' 'accelerate>=0.30' 'peft>=0.11' \
    'datasets' 'pandas<3.0' 'tqdm' 'huggingface_hub' 'sentence-transformers>=3.0' \
    'FlagEmbedding>=1.3' 'bm25s' 'lightgbm' 'scikit-learn' \
    'omegaconf' 'pyyaml' 'pylate>=1.1.0'

## Stage 1 — recall harness (nb74 parity): dev set + config-200 union pool `cs`

In [ ]:
# 3) Ensure the content-fused SASRec checkpoint exists (sasrec_v1). Trains it
# only if missing (it persists on the Drive cache across runtimes).
import os, sys
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
CACHE_DIR = '/content/recsys2026/experiments/cache'
ITEM_DB = 'talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
CORPUS = ['track_name', 'artist_name', 'album_name']
SASREC_CKPT = f'{CACHE_DIR}/retrieval_v2/sasrec/sasrec_v1/sasrec.pt'
if os.path.exists(SASREC_CKPT):
    print('[sasrec] checkpoint present, skipping train:', SASREC_CKPT)
else:
    print('[sasrec] training content-fused SASRec (sasrec_v1, ~10 epochs)...')
    !cd /content/recsys2026 && python -u scripts/train_sasrec.py \
        --cache-dir {CACHE_DIR} --out sasrec_v1 --epochs 10

In [ ]:
# 4) Build the FULL dev eval set + the config-200 union pool `cs` (verbatim nb74).
# A2: query appends listener_goal, matching the production crs_baseline query.
import numpy as np
import pandas as pd
from datasets import load_dataset
from mcrs.db_item.music_catalog import MusicCatalogDB
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.sasrec_model import build_user_dialog

item_db = MusicCatalogDB(ITEM_DB, ['all_tracks'], CORPUS)
dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')

queries, golds, user_ids, played, user_dialogs = [], [], [], [], []
goal_categories, goal_specificities, turn_numbers = [], [], []
for sess in dev:
    df = pd.DataFrame(sess['conversations'])
    goal = sess.get('conversation_goal') or {}
    goal_txt = (goal.get('listener_goal') or '').strip()  # A2: prod query includes this
    for _, music in df[df['role'] == 'music'].iterrows():
        tn = int(music['turn_number'])
        prior = df[(df['turn_number'] < tn) |
                   ((df['turn_number'] == tn) & (df['role'] == 'user'))]
        lines = []
        for _, t in prior.iterrows():
            role = 'assistant' if t['role'] == 'music' else t['role']
            content = item_db.id_to_metadata(t['content']) if t['role'] == 'music' else t['content']
            lines.append(f'{role}: {content}')
        _q = chr(10).join(lines)
        if goal_txt:
            _q = _q + chr(10) + 'goal: ' + goal_txt  # A2: train/serve parity
        queries.append(_q)
        user_dialogs.append(build_user_dialog(prior.to_dict('records')))
        golds.append(music['content'])
        user_ids.append(sess.get('user_id'))
        played.append(list(df[(df['role'] == 'music') & (df['turn_number'] < tn)]['content']))
        goal_categories.append(goal.get('category'))
        goal_specificities.append(goal.get('specificity'))
        turn_numbers.append(tn)
ctx = [{'history_tids': p, 'user_dialog': ud} for p, ud in zip(played, user_dialogs)]
print('[dev] built', len(queries), 'turns')

def recall_at(cands, k):
    return float(np.mean([1.0 if g in c[:k] else 0.0 for c, g in zip(cands, golds)]))

sas = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                            CACHE_DIR, extra_config={'use_sasrec': True, 'w_sasrec': 1.0})
cs = sas.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
print('=== config-200 union+SASRec (FULL dev, n=' + str(len(golds)) + ') ===')
print('  recall @20=' + str(round(recall_at(cs, 20), 4)) + ' @100=' + str(round(recall_at(cs, 100), 4)))

In [ ]:
# 5) Turn-1 / WALL recall@20 instrument (nb74 cell-6 helpers). The binding metric
# for Blind (100% turn-1, ~99% new-artist wall) is TURN-1 recall@20 over wall golds.
import numpy as np
from mcrs.eval_ndcg import recall_by_turn

def _artist_of(tid):
    md = item_db.metadata_dict.get(tid) or {}
    a = md.get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

is_wall = np.array([_artist_of(g) not in {_artist_of(t) for t in p}
                    for g, p in zip(golds, played)])
turn1 = np.array([t == 1 for t in turn_numbers])
print(f'[strat] turns={len(golds)} turn1={int(turn1.sum())} wall={int(is_wall.sum())} '
      f'turn1&wall={int((turn1 & is_wall).sum())}')

def _recall_mask(cands, k, mask):
    idx = np.where(mask)[0]
    if len(idx) == 0: return float('nan')
    return float(np.mean([1.0 if golds[i] in cands[i][:k] else 0.0 for i in idx]))

def strat_recall(cands, label=''):
    rep = recall_by_turn(cands, golds, turn_numbers, k=20)
    t1 = rep.get('turn1'); w1 = _recall_mask(cands, 20, turn1 & is_wall)
    print(f'  {label:34s} recall@20 overall={rep["overall"]:.4f} '
          f'turn1={(t1 if t1 is not None else float("nan")):.4f} turn1&wall={w1:.4f}')
    return rep

## Stage 2 — ColBERT zero-shot pool-rerank probe (P1)

Off-the-shelf `colbert-ir/colbertv2.0` reranks each turn-1 query's config-200
pool (`cs[i]`) by MaxSim. Doc token embeddings are built once for every track id
that appears in a turn-1 pool (no full-catalog index needed). The query fed to
ColBERT is the *same* query single-vector dense gets — so this isolates **late
interaction vs averaging**.

In [ ]:
# 82-probe) ColBERT zero-shot pool-rerank over the config-200 pool, turn-1 subset.
# Requires cell 4 (queries, cs, item_db, turn_numbers, golds) + cell 5 (masks).
# Caveat (plan §11): colbertv2.0 is MS-MARCO/web-trained; a flat result conflates
# 'late interaction' with domain shift — Phase 2 fine-tune disambiguates. A clear
# win here is decisive; a flat result is a soft (not hard) kill.
import numpy as np
from mcrs.retrieval_modules.colbert_late import (ColbertRetriever, DEFAULT_COLBERT_MODEL,
                                                 strip_track_id_prefix)

# Restrict the probe to turn-1 rows (the Blind proxy; plan §9 P1).
t1_idx = [i for i, t in enumerate(turn_numbers) if t == 1]
print(f'[probe] turn-1 rows: {len(t1_idx)}')

# Build ColBERT doc token-embeddings ONCE for every tid in a turn-1 pool.
pool_tids = sorted({tid for i in t1_idx for tid in cs[i]})
# Strip the leading 'track_id: <uuid>' (RCA #4: UUID hex dilutes MaxSim).
pool_texts = [strip_track_id_prefix(item_db.id_to_metadata(t)) for t in pool_tids]
print(f'[probe] unique pool docs to encode: {len(pool_tids)}')

retr = ColbertRetriever(doc_embs={}, model_name=DEFAULT_COLBERT_MODEL, q_len=96, d_len=96)
retr.encode_docs(pool_tids, pool_texts)  # one-time GPU encode (~minutes)

q_t1 = [queries[i] for i in t1_idx]
pools_t1 = [cs[i] for i in t1_idx]
colbert_t1 = retr.batch_rerank_pool(q_t1, pools_t1, topk=100)

# Splice reranked turn-1 lists back into a full-length cands array (non-turn-1 rows
# keep cs ordering) so the cell-5 strat_recall helpers apply unchanged.
colbert_cands = list(cs)
for j, i in enumerate(t1_idx):
    colbert_cands[i] = colbert_t1[j]
print('[probe] reranked', len(t1_idx), 'turn-1 pools')

# Reranker headroom: recall@20 can never exceed the pool's recall@100 (RCA #3).
print(f'[probe] turn-1&wall cs recall@100 (rerank CEILING) = {_recall_mask(cs, 100, turn1 & is_wall):.4f}  recall@20 = {_recall_mask(cs, 20, turn1 & is_wall):.4f}')

In [ ]:
# 82-gate) KILL-GATE: ColBERT vs single-vector dense vs union on turn-1 wall recall@20.
# Dense baseline = the dense_metadata_qwen3_instruct channel standalone (the single
# vector ColBERT must beat). Union (cs) = current config-200 ordering. ColBERT only
# reranks the cs pool, so its recall@100 == cs's by construction; the lift is purely
# @20 (in-pool golds in the rank 21-100 band lifted into top-20 — plan §0).
dense_only = load_retrieval_module('dense_metadata_qwen3_instruct', ITEM_DB, ['all_tracks'],
                                   CORPUS, CACHE_DIR, extra_config={})
dense_cands = dense_only.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids)
# (dense is content-only; it takes no batch_context, unlike the RRF union)

print('=== TURN-1 WALL RECALL@20 GATE (n_turn1=' + str(int(turn1.sum())) +
      ', n_turn1&wall=' + str(int((turn1 & is_wall).sum())) + ') ===')
for label, cands in [('single-vector dense', dense_cands),
                     ('union+SASRec (cs)', cs),
                     ('ColBERT rerank (cs pool)', colbert_cands)]:
    strat_recall(cands, label)

# Verdict on the binding metric.
dense_w = _recall_mask(dense_cands, 20, turn1 & is_wall)
cb_w = _recall_mask(colbert_cands, 20, turn1 & is_wall)
print()
print(f'[gate] turn1&wall recall@20: dense={dense_w:.4f}  colbert={cb_w:.4f}  '
      f'delta={cb_w - dense_w:+.4f}')
if cb_w > dense_w:
    print('[gate] PASS — late interaction beats single-vector dense. Proceed to P2',
          '(fine-tune music-colbert-v1 + PLAID index).')
else:
    print('[gate] FAIL/SOFT — see caveat (domain shift). If Phase-2 fine-tune is not',
          'at least neutral, STOP and bank config 200 (plan §9 kill-rule).')

## Stage 3 — P2: fine-tune `music-colbert-v1` + re-probe (tightened gate ladder)

P1 was a soft pass (ColBERT 0.258 > dense 0.215, but < union 0.325). P2 pays the
domain-shift tax: fine-tune on train-split MOVES_TOWARD_GOAL triples, then re-run
the **exact same pool-rerank probe**. **New bar = beat the union's 0.325** (not
dense's 0.215). Build the expensive PLAID full-catalog index only if this clears it.

> **RCA fixes applied (2026-06-11, wf8ii499r) — the earlier 0.281<0.325 was an INVALID kill-test.** Now: (1) turn-1 training positives are synthesized (were 100% filtered out by MOVES_TOWARD_GOAL); (2) q_len=96 keeps the `goal:` facet (q_len=32 truncated it off ~95% of queries); (3) doc text strips the `track_id:` UUID prefix. The reprobe is now a *fair* rerank test — but a rerank win/loss still isn't the recall thesis (that needs Stage B: the PLAID full-catalog index).

In [ ]:
# 82-build-data) P2 step 1: build ColBERT fine-tune triples (TRAIN split,
# MOVES_TOWARD_GOAL only, hard negs = SASRec-free pool non-golds). One-time;
# persists to the Drive cache. Use --max-rows 2000 first for a smoke build.
TRAIN_JSONL = f'{CACHE_DIR}/retrieval_v2/colbert_train.jsonl'
!cd /content/recsys2026 && python -u scripts/build_colbert_train_data.py \
    --output {TRAIN_JSONL} --cache-dir {CACHE_DIR} --pool-size 100 --k-negs 15 --max-rows 0
import os
print('[build-data] exists:', os.path.exists(TRAIN_JSONL),
      '| lines:', sum(1 for _ in open(TRAIN_JSONL)) if os.path.exists(TRAIN_JSONL) else 0)

In [ ]:
# 82-dev-eval-pack) Build the dev validation pack the trainer selects on. SAME
# turn-1 pool/golds as #82-reprobe, so in-loop model selection optimizes the EXACT
# gate metric (never a train-internal loss - the campaign's anti-overfit rule).
# Requires cells 4,5,7 (q_t1, pools_t1, t1_idx, golds, is_wall, item_db).
import pickle, os
from mcrs.retrieval_modules.colbert_late import strip_track_id_prefix
golds_t1 = [golds[i] for i in t1_idx]
wall_t1 = [bool(is_wall[i]) for i in t1_idx]
need = sorted({t for pool in pools_t1 for t in pool})
pack = {'queries': q_t1, 'pools': pools_t1, 'golds': golds_t1, 'wall': wall_t1,
        'tid_to_text': {t: strip_track_id_prefix(item_db.id_to_metadata(t)) for t in need}}
DEV_PACK = f'{CACHE_DIR}/retrieval_v2/colbert_dev_eval.pkl'
os.makedirs(os.path.dirname(DEV_PACK), exist_ok=True)
pickle.dump(pack, open(DEV_PACK, 'wb'))
print('[dev-pack]', len(pack['queries']), 'turn-1 queries,', len(need), 'pool docs ->', DEV_PACK)

In [ ]:
# 82-finetune) P2 step 2: warm-start colbertv2.0 -> music-colbert-v1 (PyLate
# contrastive). Validates on the dev pack every --eval-steps and saves the SINGLE
# best checkpoint by turn-1 recall@20 (overrides). --epochs 2 gives selection room.
COLBERT_OUT = f'{CACHE_DIR}/retrieval_v2/colbert/music-colbert-v1'
!cd /content/recsys2026 && python -u scripts/train_colbert.py \
    --train-jsonl {TRAIN_JSONL} --dev-eval-pack {DEV_PACK} --out-dir {COLBERT_OUT} \
    --epochs 2 --batch-size 32 --eval-steps 500 --dev-subset 300
print('[finetune] best model saved to', COLBERT_OUT)

In [ ]:
# 82-reprobe) P2 GATE: re-run the EXACT pool-rerank probe with FINE-TUNED
# music-colbert-v1. New bar = beat the UNION's turn-1 wall recall@20, not just
# dense. Reuses cell-5 masks + #82-probe vars (pool_tids, pool_texts, q_t1,
# pools_t1, t1_idx, colbert_cands). Build the PLAID index ONLY if this clears the union.
retr_ft = ColbertRetriever(doc_embs={}, model_name=COLBERT_OUT, q_len=96, d_len=96)
retr_ft.encode_docs(pool_tids, pool_texts)   # same turn-1 pool docs as zero-shot
colbert_ft_t1 = retr_ft.batch_rerank_pool(q_t1, pools_t1, topk=100)
colbert_ft_cands = list(cs)
for j, i in enumerate(t1_idx):
    colbert_ft_cands[i] = colbert_ft_t1[j]

print('=== P2 RE-PROBE: fine-tuned vs union vs zero-shot (turn-1 wall recall@20) ===')
for label, cands in [('union+SASRec (cs)', cs),
                     ('ColBERT zero-shot', colbert_cands),
                     ('ColBERT fine-tuned', colbert_ft_cands)]:
    strat_recall(cands, label)

union_w = _recall_mask(cs, 20, turn1 & is_wall)
ft_w = _recall_mask(colbert_ft_cands, 20, turn1 & is_wall)
print()
print(f'[P2 gate] turn1&wall recall@20: union={union_w:.4f}  colbert_ft={ft_w:.4f}  '
      f'delta={ft_w - union_w:+.4f}')
if ft_w > union_w:
    print('[P2 gate] PASS - fine-tuned ColBERT beats the union it reranks. Build the',
          'PLAID full-catalog index (P2 step 3): it can surface NEW golds, not just reorder.')
else:
    print('[P2 gate] FAIL (rerank) - fine-tuned ColBERT below the union on the FAIR test.',
          'Reranking is not the win; the recall thesis still needs Stage B (build the PLAID',
          'full-catalog index, gate on union recall@100 vs 0.495). Only a flat full-catalog',
          'recall@100 justifies banking config 200.')